# seminal quickstart

`seminal` implements the two azoospermia prediction models of Tradewell et al.
(World J Mens Health 2022;40:600-607, doi:10.5534/wjmh.210138) with the
coefficients exactly as printed.

Both models estimate the probability that a man presenting for infertility
evaluation has no sperm in the ejaculate, without a semen analysis.

* **FSH-only**: a quadratic in serum FSH (Fig. 1 of the paper)
* **Logistic regression**: age, FSH, LH, total testosterone and mean testis
  volume (Table 3 of the paper)

It also carries the WHO 2021 semen reference distributions (Campbell et al.,
Andrology 2021), the fertile-man centiles behind the 6th-edition laboratory
manual.

This notebook runs both azoospermia models on one patient, then on the
authors' own validation cohorts to show the implementation reproduces the
published results, and ends with the semen reference functions.

In [ ]:
# pip install seminal   (or `make install` from the repo root for the editable version)
import seminal
from seminal import azoospermia_fsh, azoospermia_lr, add_azoospermia_risk_cols

print(seminal.__version__)

## One patient

The median man from the paper's Table 1: age 36, FSH 5.7 IU/L, LH 4.5 IU/L,
total testosterone 429 ng/dL, mean testis volume 15 mL.

The FSH-only model needs just the FSH. The logistic model needs all five
inputs. Both return a probability between 0 and 1.

In [ ]:
patient = dict(age=36, fsh=5.7, lh=4.5, testosterone=429, testis_volume=15)

p_fsh = azoospermia_fsh(patient["fsh"])  # FSH-only quadratic
p_lr = azoospermia_lr(**patient)  # full logistic model

print(f"FSH-only model:   {p_fsh:.1%}")
print(f"Logistic model:   {p_lr:.1%}")

# Push FSH up and watch both models respond
for fsh in [2, 5.7, 10, 15, 20, 30]:
    print(
        f"FSH {fsh:>5}  ->  FSH-only {azoospermia_fsh(fsh):>6.1%}   "
        f"logistic {azoospermia_lr(**{**patient, 'fsh': fsh}):>6.1%}"
    )

## The FSH curve

The paper describes a U-shaped relationship: risk is 10.1% at undetectable
FSH, lowest (about 8%) near 3.6 IU/L, and effectively certain above 30 IU/L.
Plot the FSH-only model across the clinical range and mark those three
anchors. Overlay the logistic model with the other inputs held at the median
patient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fsh = np.linspace(0, 40, 401)
p_quad = azoospermia_fsh(fsh)  # vectorized
p_logit = [azoospermia_lr(**{**patient, "fsh": f}) for f in fsh]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fsh, p_quad, label="FSH-only quadratic (Fig. 1)")
ax.plot(fsh, p_logit, label="Logistic model, other inputs at median", linestyle="--")

# The paper's three anchors
anchors = [
    (0, "10.1% at FSH 0", 8, 10),
    (3.6, "nadir at 3.6", 8, -16),
    (30, "100% at ~30", 8, -14),
]
ax.scatter(
    [a[0] for a in anchors],
    [azoospermia_fsh(a[0]) for a in anchors],
    zorder=3,
    color="black",
)
for x, txt, dx, dy in anchors:
    ax.annotate(
        txt, (x, azoospermia_fsh(x)), xytext=(dx, dy), textcoords="offset points"
    )

ax.set_xlabel("Serum FSH (IU/L)")
ax.set_ylabel("Probability of azoospermia")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()

## Scoring a cohort

The study data are public (CC0) on Harvard Dataverse. `make data_gen preprocess`
downloads them and builds `data/processed/tradewell_2022.parquet`: one row per
semen analysis with the five model inputs, the azoospermia outcome, the site,
and an `included` flag reproducing the paper's cohort (Miami 946, Milan 1,955,
Chicago 596).

Score every row with both models and compare discrimination per site to what
the paper reported: Milan AUC 0.79 / 0.78 and Chicago 0.72 / 0.71
(logistic / FSH-only). Miami is the development cohort, so its AUC is
in-sample.

In [ ]:
import pandas as pd
from model_metrics import summarize_model_performance

df = pd.read_parquet("../data/processed/tradewell_2022.parquet")
scored = add_azoospermia_risk_cols(df[df["included"]])

# Per site, both models side by side
for site in ["miami", "milan", "chicago"]:
    g = scored[scored["site"] == site]
    print(f"\n{site}  (n={len(g)}, azoospermic={int(g['azoospermia'].sum())})")
    summarize_model_performance(
        y_prob=[g["azoo_lr"], g["azoo_fsh"]],
        y=g["azoospermia"],
        model_title=["Logistic (Table 3)", "FSH-only (Fig. 1)"],
    )

ROC, precision-recall and calibration on the two external sites in one figure.
AUCs match the paper. The calibration panel is what the paper did not show:
the published coefficients applied as-is to another clinic, without refitting.

In [ ]:
from model_metrics import (
    show_roc_curve,
    show_pr_curve,
    show_calibration_curve,
    combine_plots,
)

milan = scored[scored["site"] == "milan"]
chicago = scored[scored["site"] == "chicago"]

# Shared inputs for every panel: four curves, two sites x two models
curves = dict(
    y_prob=[
        milan["azoo_lr"],
        milan["azoo_fsh"],
        chicago["azoo_lr"],
        chicago["azoo_fsh"],
    ],
    y=[milan["azoospermia"]] * 2 + [chicago["azoospermia"]] * 2,
    model_title=[
        "Milan logistic",
        "Milan FSH-only",
        "Chicago logistic",
        "Chicago FSH-only",
    ],
    overlay=True,
)

combine_plots(
    [
        (show_roc_curve, {**curves, "title": "ROC"}),
        (show_pr_curve, {**curves, "title": "Precision-recall"}),
        (show_calibration_curve, {**curves, "title": "Calibration"}),
    ],
    n_cols=3,
    figsize=(18, 5.5),
    suptitle="External validation, Tradewell 2022 (coefficients as published)",
    image_filename="../assets/external_validation.svg",
)

## Explaining a prediction

`azoospermia_lr(..., explain=True)` returns the linear predictor and each
input's contribution to the log-odds, largest first. Contributions are raw
coefficient × value, so they carry each variable's scale (testosterone in
ng/dL is a small coefficient on a large number).

In [ ]:
high_risk = dict(age=40, fsh=18, lh=9, testosterone=280, testis_volume=9)
out = azoospermia_lr(**high_risk, explain=True)

print(
    f"risk = {out['risk']:.1%}   "
    f"(log-odds {out['linear_predictor']:.2f}, intercept {out['intercept']})"
)
for k, v in out["contributions"].items():
    print(f"  {k:<14} {v:+.2f}")

# Same man with FSH back in the normal range: how much of the risk was FSH?
normal_fsh = azoospermia_lr(**{**high_risk, "fsh": 5})
print(f"\nwith FSH 5 instead of 18: {normal_fsh:.1%}")

## WHO 2021 semen reference

`semen_percentile` places a result among fertile men (interpolated between
the nine published centiles, clipped to 2.5 to 97.5); `lower_reference_limit`
is the WHO 5th centile. Units are mL, 10^6/mL, 10^6 per ejaculate and
percentages on a 0 to 100 scale; morphology is strict criteria.

In [ ]:
from seminal import (
    semen_percentile,
    lower_reference_limit,
    below_reference,
    semen_reference,
    add_semen_percentile_cols,
)

print("concentration 36 x10^6/mL:", semen_percentile("concentration", 36), "th centile")
print("lower reference limit:    ", lower_reference_limit("concentration"))
print("below limit at 12:        ", below_reference("concentration", 12))
semen_reference()[["label", "n", "p5", "p50", "p95"]]

For a cohort, `add_semen_percentile_cols` maps parameter names to your
column names and adds a `_pct` percentile and a `_below_lrl` flag per
parameter. Compute total sperm number as volume times concentration if it is
not recorded.

In [ ]:
analyses = pd.DataFrame(
    {
        "vol_ml": [3.2, 1.1, 2.5],
        "conc": [45, 8, 120],
        "pr": [40, 20, 65],
        "morph": [5, 2, 9],
    }
)
analyses["tsc"] = analyses["vol_ml"] * analyses["conc"]

add_semen_percentile_cols(
    analyses,
    columns={
        "volume": "vol_ml",
        "concentration": "conc",
        "total_number": "tsc",
        "progressive_motility": "pr",
        "normal_forms": "morph",
    },
)

## Where to go next

* `01_reproduce_tradewell.ipynb`: reconstruct the paper's cohort and Table 1,
  refit the logistic model on the Miami rows, and recalibrate per site.
* `02_reproduce_campbell.ipynb`: recompute the WHO 2021 centiles from the
  authors' deposit and score a cohort against them.
* Both models are implemented exactly as published. They are research tools,
  not clinical decision aids.